<a href="https://colab.research.google.com/github/dnltjrqja-dev/PPjugglingwithMPC/blob/main/mainnote.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 매번 하는 Colab 루틴

**첫 셀 (코랩 켤 때마다 실행)**


- 같은 런타임에서 두 번 돌려서 "폴더 이미 있음" 에러 나면 `%cd PPjugglingwithMPC`부터 실행
- `!cd` 말고 `%cd` 써야 폴더 이동이 유지됨


In [29]:
import os
from google.colab import userdata
!git clone -q https://dnltjrqja-dev@github.com/dnltjrqja-dev/PPjugglingwithMPC.git
%cd PPjugglingwithMPC
!git config user.name "dnltjrqja-dev" && git config user.email "dnltjrqja@gmail.com"

/content/PPjugglingwithMPC/PPjugglingwithMPC/PPjugglingwithMPC


- 같은 런타임에서 두 번 돌려서 "폴더 이미 있음" 에러 나면 `%cd PPjugglingwithMPC`부터 실행
- `!cd` 말고 `%cd` 써야 폴더 이동이 유지됨

**단계 하나 끝날 때마다 / 퇴근할 때 (삼총사)**

In [23]:
!git add -A && git commit -m "0단계: 공 시뮬레이션" && git push

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


- `/content` 안의 파일은 런타임 끝나면 사라짐. push한 것만 살아남음

---

# 0단계: 준비
**목표**: Colab에서 MuJoCo 설치, 바닥 위에 공 하나 떨어지는 장면을 영상으로 보기

**질문**: MjModel과 MjData는 각각 무엇을 담나? 화면 없는 Colab에서 렌더링하려면 뭘 설정해야 하나?

**통과 기준**: 공 떨어지는 영상이 노트북 안에 나온다

In [24]:
!pip install mujoco

In [25]:
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
import mujoco
print(mujoco.__version__)

4.0.0


In [26]:
xml = """ <mujoco>
  <worldbody>
    <light pos="0 0 3"/>
    <geom type="plane" size="1 1 0.1"/>
    <body name="sphere" pos="0 0 0.5">
      <geom type="sphere" size="0.02" rgba="1 0 0 1"/>
      <joint type="free"/>
    </body>
  </worldbody>
</mujoco> """
model = mujoco.MjModel.from_xml_string(xml)
data = mujoco.MjData(model)

In [27]:
!pip install mediapy
import mediapy as media

In [28]:
mujoco.mj_resetData(model, data)
renderer = mujoco.Renderer(model, 480, 640)
N = int(1/model.opt.timestep)
rollout = []
for n in range(N):
  mujoco.mj_step(model, data)
  if n % int(N/30) == 0:
    renderer.update_scene(data)
    rollout.append(renderer.render())
    print(data.body("sphere").xpos)
media.show_video(rollout, fps = 30)

[0.  0.  0.5]
[0.         0.         0.49466336]
[0.         0.         0.47928128]
[0.         0.         0.45385376]
[0.        0.        0.4183808]
[0.        0.        0.3728624]
[0.         0.         0.31729856]
[0.         0.         0.25168928]
[0.         0.         0.17603456]
[0.        0.        0.0903344]
[0.         0.         0.00534891]
[0.         0.         0.00681314]
[0.         0.         0.01511743]
[0.         0.         0.01833207]
[0.         0.         0.01935575]
[0.         0.         0.01960859]
[0.         0.         0.01963624]
[0.         0.         0.01963445]
[0.         0.         0.01963311]
[0.         0.         0.01963284]
[0.         0.         0.01963281]
[0.         0.         0.01963282]
[0.         0.         0.01963282]
[0.         0.         0.01963282]
[0.         0.         0.01963282]
[0.         0.         0.01963282]
[0.         0.         0.01963282]
[0.         0.         0.01963282]
[0.         0.         0.01963282]
[0.         0. 

## 1단계: 공의 물리 — 튀는 성질 (가장 중요)
- **목표:** 지름 40 mm, 질량 2.7 g 공이 실제 탁구공처럼 튀게 만들기
- **질문:**
  - MuJoCo 접촉은 "딱딱한 충돌"이 아니라 무엇으로 모델링되나? `solref`, `solimp` 문서 읽기
  - `solref`를 양수로 쓸 때와 음수로 쓸 때 의미 차이는?
  - 감쇠 진동 모델에서 반발계수 e와 감쇠비 ζ의 관계를 직접 유도
  - 접촉 지속 시간과 timestep은 어떤 관계여야 하나?
  - 공기저항이 없으면 튀어 오른 높이와 반발계수 e는 어떤 관계인가? 직접 유도
- **통과 기준:**
  - 30.5 cm에서 떨어뜨리면 23~25 cm 튀어 오른다 (실제 탁구공 규정 테스트 기준)
  - 반발계수 측정 코드를 직접 짠다 (충돌 직전 속도와 직후 속도의 비)
  - 측정한 e로 계산한 높이와 실제로 튀어 오른 높이가 맞는다